# People Like Me

Setup:  create a .venv -- must be in a workspace to do this.  This .venv will be available to all projects in workspace
Use:  https://chatgpt.com/share/69cd1090-2a98-8328-9f28-b24fa7b362ce

In [ ]:
# import kagglehub

# # Download latest version
# path = kagglehub.dataset_download("snehaanbhawal/resume-dataset")

# print("Path to dataset files:", path)

#### For HuggingFace, install library "datasets"

In [1]:
# From Hugging Face:
# pip install datasets
# Dataset is located here:  C:\Users\DeloresMincarelli\.cache\huggingface\hub\

import pandas as pd
from datasets import load_dataset

dataset = load_dataset("med2425/resume-job-fit-merged-v1")

In [11]:
import json

row = dataset["train"][0]
print(json.dumps(row, indent=2, ensure_ascii=False))

{
  "resume": "SummarySelf-motivated accountant  offering a strong work ethic and determination to complete tasks in a timely manner. Accurate and detail-oriented with extensive auditing and finance knowledge.\nHighlightsComplex problem solvingStrong communication skillsExpert in customer relationsPortfolio managementAProficient in Microsoft OfficeMicrosoft Excel expertRisk management expertiseFinancial statement analysisGeneral ledger accounting\nExperienceCurrentto08/2014AccountantApartment Income Reit Corp.–Nashua,NH,Prepare unpaid reports on actual expenses for marketing line of business.Create and maintain pending and process able database.Prepare and setup vendor purchase orders contracts as well as CRX templates.Verify funding and SAP project code against the most recent budget/forecast submission.Key invoices into ePurchase system as well as approve and reconcile invoices.Track invoices from submission to payment on database.Monitor invoice central mailbox that will include inv

In [12]:

train = dataset["train"].to_pandas()
test = dataset["test"].to_pandas()

train["Is_Me"] = 0
test["Is_Me"] = 0

train.head()
# row = train.iloc[0].to_dict()
# print(json.dumps(row, indent=2, ensure_ascii=False))

,resume,jd,label,source,resume_domain,jd_domain,Is_Me
0,SummarySelf-motivated accountant offering a s...,"Our client, a Raleigh-based full-service comme...",Potential Fit,generated_smart,finance,finance,0
1,Professional ProfileSpecialized knowledge of r...,"""All candidates must be directly contracted by...",No Fit,generated_smart,finance,software,0
2,"SummaryEngineering Project Manager\nAmbitious,...",Calling all innovators find your future at Fi...,No Fit,generated_smart,software,software,0
3,SummaryData Protection Consultant with 10 year...,"MUST WORK ON A W2 INDEPENDENTLY, NO SPONSORSHI...",No Fit,generated_smart,software,data,0
4,Career OverviewHighly skilled SOFTWARE QUALITY...,Calling all data nerds who love finance! \nIf ...,No Fit,generated_smart,software,data,0


In [13]:
distinct_resume_domains = sorted(train["resume_domain"].dropna().unique())
distinct_resume_domains

['ai',
 'data',
 'design',
 'engineering',
 'finance',
 'healthcare',
 'hr',
 'legal',
 'management',
 'marketing',
 'sales',
 'software']

In [14]:
# Cleaning

if "Is_Me" not in train.columns:
    train["Is_Me"] = 0
if "Is_Me" not in test.columns:
    test["Is_Me"] = 0

train_resume = train[["resume", "resume_domain", "source", "Is_Me"]].copy()
test_resume = test[["resume", "resume_domain", "source", "Is_Me"]].copy()

train_resume = train_resume.drop_duplicates(subset=["resume"], keep="first").reset_index(drop=True)
test_resume = test_resume.drop_duplicates(subset=["resume"], keep="first").reset_index(drop=True)

# One ID per distinct resume text (shared across train/test)
combined_resume = pd.concat([train_resume["resume"], test_resume["resume"]], ignore_index=True)
resume_ids, _ = pd.factorize(combined_resume)
n_train = len(train_resume)
train_resume["resume_id"] = resume_ids[:n_train]
test_resume["resume_id"] = resume_ids[n_train:]

train_resume = train_resume[["resume_id", "resume", "resume_domain", "source", "Is_Me"]]
test_resume = test_resume[["resume_id", "resume", "resume_domain", "source", "Is_Me"]]

train_resume.head()

,resume_id,resume,resume_domain,source,Is_Me
0,0,SummarySelf-motivated accountant offering a s...,finance,generated_smart,0
1,1,Professional ProfileSpecialized knowledge of r...,finance,generated_smart,0
2,2,"SummaryEngineering Project Manager\nAmbitious,...",software,generated_smart,0
3,3,SummaryData Protection Consultant with 10 year...,software,generated_smart,0
4,4,Career OverviewHighly skilled SOFTWARE QUALITY...,software,generated_smart,0


### Add your resume from a PDF or `my_resume_extracted.txt`

Install once: `pip install pypdf ipywidgets`

1. Run the next cell (extract helpers).
2. Run the cell after that (optional path + upload widget). If you use **Upload**, pick your PDF, then run the **following** cell again so the widget value is read.
3. Run the last cell: it prefers PDF (upload or `dam_resume.pdf`) if available, otherwise loads `my_resume_extracted.txt`. It removes any prior `source == "me"` rows, then appends your resume to **both** `train` and `test` with `resume_domain='data'`, `source='me'`, `Is_Me=1`. Then **re-run** the Cleaning cell to refresh `train_resume` / `test_resume`.

In [6]:
import io
import re
from pathlib import Path

from pypdf import PdfReader


def organize_resume_text(text: str) -> str:
    """Normalize whitespace and page breaks for cleaner plain text."""
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t\f\v]+", " ", text)
    text = re.sub(r"\n[ \t]+", "\n", text)
    text = re.sub(r"[ \t]+\n", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def extract_text_from_pdf(data: bytes) -> str:
    reader = PdfReader(io.BytesIO(data))
    parts = []
    for page in reader.pages:
        t = page.extract_text()
        if t:
            parts.append(t.strip())
    return organize_resume_text("\n\n".join(parts))


print("Ready: organize_resume_text, extract_text_from_pdf")

Ready: organize_resume_text, extract_text_from_pdf


### Use Upload button to upload pdf resume

In [7]:
import ipywidgets as widgets
from IPython.display import display

NOTEBOOK_DIR = Path.cwd()
# PDF in the same folder as the notebook (when Jupyter cwd is this folder)
MY_RESUME_PDF = NOTEBOOK_DIR / "dam_resume.pdf"

print(f"If no upload button appears (common in Cursor), use a PDF at:\n  {MY_RESUME_PDF.resolve()}\n")

pdf_upload = widgets.FileUpload(accept=".pdf", multiple=False, description="Resume PDF")
display(pdf_upload)

If no upload button appears (common in Cursor), use a PDF at:
  C:\Users\DeloresMincarelli\Documents\Portfolio\peopleLikeMe\dam_resume.pdf



FileUpload(value=(), accept='.pdf', description='Resume PDF')

### Scrape PDF text and save as `my_resume_extracted.txt`

In [8]:
RESUME_TEXT_CACHE = NOTEBOOK_DIR / "my_resume_extracted.txt"


def _pdf_bytes_from_upload(w) -> bytes | None:
    if not w.value:
        return None
    f0 = w.value[0]
    content = f0["content"]
    if hasattr(content, "tobytes"):
        return content.tobytes()
    return bytes(memoryview(content))


pdf_bytes = _pdf_bytes_from_upload(pdf_upload)
if pdf_bytes is not None:
    print("Using PDF from upload widget.")
elif MY_RESUME_PDF.exists():
    pdf_bytes = MY_RESUME_PDF.read_bytes()
    print(f"Using PDF from disk: {MY_RESUME_PDF.resolve()}")
else:
    raise FileNotFoundError(
        f"No PDF yet: use the upload widget and re-run this cell, "
        f"or save your file as:\n  {MY_RESUME_PDF.resolve()}"
    )

my_resume_text = extract_text_from_pdf(pdf_bytes)
RESUME_TEXT_CACHE.write_text(my_resume_text, encoding="utf-8")

print(f"Saved text to {RESUME_TEXT_CACHE} ({len(my_resume_text):,} chars).")

Using PDF from upload widget.
Saved text to c:\Users\DeloresMincarelli\Documents\Portfolio\peopleLikeMe\my_resume_extracted.txt (5,427 chars). train rows: 80018


In [22]:
# Insert my resume into train_resume and test_resume
my_resume_text = (NOTEBOOK_DIR / "my_resume_extracted.txt").read_text(encoding="utf-8")

# Remove any prior "me" rows so re-running doesn't duplicate
train_resume = train_resume[train_resume["source"] != "me"].reset_index(drop=True)
test_resume = test_resume[test_resume["source"] != "me"].reset_index(drop=True)

next_id = train_resume["resume_id"].max() + 1

new_row = pd.DataFrame(
    [
        {
            "resume_id": next_id,
            "resume": my_resume_text,
            "resume_domain": "data",
            "source": "me",
            "Is_Me": 1,
        }
    ]
)
train_resume = pd.concat([train_resume, new_row], ignore_index=True)
test_resume = pd.concat([test_resume, new_row], ignore_index=True)

print(
    f"Resume text: {len(my_resume_text):,} chars. "
    f"train_resume rows: {len(train_resume)}, test_resume rows: {len(test_resume)}"
)

# Verify
train_resume[train_resume["Is_Me"] == 1]

Resume text: 5,480 chars. train_resume rows: 643, test_resume rows: 478


,resume_id,resume,resume_domain,source,Is_Me
642,642,SUMMARY\nPurpose-driven healthcare data scient...,data,me,1


### POC: Extract resume section headings

In [23]:
import re

my_resume_df = train_resume[train_resume["Is_Me"] == 1].copy()

if len(my_resume_df) != 1:
    raise ValueError("Expected exactly 1 row where Is_Me == 1")

text = my_resume_df.iloc[0]["resume"]

lines = [line.strip() for line in text.splitlines()]
lines = [line for line in lines if line]

candidate_headings = []

known_headings = {
    "summary",
    "professional summary",
    "experience",
    "work experience",
    "professional experience",
    "education",
    "skills",
    "technical skills",
    "projects",
    "certifications",
    "awards",
}

for line in lines:
    alpha = re.sub(r"[^A-Za-z]", "", line)

    uppercase_ratio = (
        sum(1 for c in alpha if c.isupper()) / len(alpha)
        if len(alpha) > 0 else 0
    )

    is_short = len(line) <= 40
    is_known = line.lower() in known_headings
    looks_upper = line.isupper() or uppercase_ratio > 0.8

    if is_short and (is_known or looks_upper):
        candidate_headings.append(line)

candidate_headings = list(dict.fromkeys(candidate_headings))

print("Candidate headings:")
for h in candidate_headings:
    print("-", h)

Candidate headings:
- SUMMARY
- EXPERIENCE
- SKILLS
- EDUCATION
- CERTIFICATIONS
- PUBLICATIONS


In [24]:
heading_map = {
    "SUMMARY": "Summary",
    "EXPERIENCE": "Work Experience",
    "SKILLS": "Skills",
    "EDUCATION": "Education",
    "CERTIFICATIONS": "Certifications",
    "PUBLICATIONS": "Publications",
}

heading_map

{'SUMMARY': 'Summary',
 'EXPERIENCE': 'Work Experience',
 'SKILLS': 'Skills',
 'EDUCATION': 'Education',
 'CERTIFICATIONS': 'Certifications',
 'PUBLICATIONS': 'Publications'}

In [26]:
my_resume_df = train_resume[train_resume["Is_Me"] == 1].copy()

if len(my_resume_df) != 1:
    raise ValueError("Expected exactly 1 row where Is_Me == 1")

resume_id = my_resume_df.iloc[0]["resume_id"]
text = my_resume_df.iloc[0]["resume"]

lines = [line.strip() for line in text.splitlines()]
lines = [line for line in lines if line]

rows = []
current_section = "Unknown"

for line in lines:
    if line in heading_map:
        current_section = heading_map[line]
        continue

    rows.append({
        "resume_id": resume_id,
        "section": current_section,
        "line_text": line
    })

sections_df = pd.DataFrame(rows)

print(sections_df.head(80).to_string(index=False))
sections_df.to_csv("resume_lines_by_section.csv", index=False)
print("Saved resume_lines_by_section.csv")

 resume_id         section                                                                                                                 line_text
       642         Summary     Purpose-driven healthcare data scientist who partners closely with clinicians and managers to improve decision-making
       642         Summary           through thoughtful analytics. Comfortable tackling routine, messy, and complex problems alike—finding scalable,
       642         Summary      repeatable solutions that strengthen systems over time. I bring strong critical thinking, clear communication, and a
       642         Summary                           learning mindset to every engagement, using AI as a tool rather than a substitute for judgment.
       642 Work Experience                                                                                             Senior Implementation Analyst
       642 Work Experience                                                                                

### Extract work experience bullets

In [27]:
my_resume_df = train_resume[train_resume["Is_Me"] == 1].copy()

if len(my_resume_df) != 1:
    raise ValueError("Expected exactly 1 row where Is_Me == 1")

resume_id = my_resume_df.iloc[0]["resume_id"]
text = my_resume_df.iloc[0]["resume"]

lines = [line.strip() for line in text.splitlines()]
lines = [line for line in lines if line]

rows = []
current_section = "Unknown"
bullet_id = 1

for line in lines:
    if line in heading_map:
        current_section = heading_map[line]
        continue

    if current_section != "Work Experience":
        continue

    is_bullet = (
        line.startswith("-")
        or line.startswith("•")
        or re.match(r"^\u2022", line) is not None
    )

    if is_bullet:
        bullet_text = re.sub(r"^[\-\•\u2022]\s*", "", line)

        rows.append({
            "resume_id": resume_id,
            "bullet_id": bullet_id,
            "section": current_section,
            "bullet_text": bullet_text,
            "Is_Me": 1
        })
        bullet_id += 1

resume_bullets_df = pd.DataFrame(rows)

print(resume_bullets_df.to_string(index=False))
resume_bullets_df.to_csv("resume_bullets.csv", index=False)
print("Saved resume_bullets.csv")

 resume_id  bullet_id         section                                                                                                          bullet_text  Is_Me
       642          1 Work Experience      LLM Fine-Tuning & Evaluation (Healthcare Analytics): Fine Tuned a LLM on laboratory data, and evaluated against      1
       642          2 Work Experience     Created an automated process using Cursor AI to interpret complex SQL code and craft operational definitions and      1
       642          3 Work Experience     Innovative Problem Solving in Analytics: Developed and implemented one robust database design to store customer-      1
       642          4 Work Experience   Collaborative Leadership in Technical Projects: Led end-to-end database releases, including overall coordination ,      1
       642          5 Work Experience         Data-Driven Insights and Quality Improvements: Analyzed and resolved complex bugs, conducted deep dives into      1
       642          6 Work E

### Load O*NET data files
https://www.onetcenter.org/database.html#all-files

In [29]:
onet_dir = Path("data/onet")

occupation_df = pd.read_csv(onet_dir / "Occupation Data.txt", sep="\t", dtype=str)
tasks_df = pd.read_csv(onet_dir / "Task Statements.txt", sep="\t", dtype=str)
skills_df = pd.read_csv(onet_dir / "Skills.txt", sep="\t", dtype=str)

print("Occupation Data columns:")
print(occupation_df.columns.tolist())
print()

print("Task Statements columns:")
print(tasks_df.columns.tolist())
print()

print("Skills columns:")
print(skills_df.columns.tolist())

Occupation Data columns:
['O*NET-SOC Code', 'Title', 'Description']

Task Statements columns:
['O*NET-SOC Code', 'Task ID', 'Task', 'Task Type', 'Incumbents Responding', 'Date', 'Domain Source']

Skills columns:
['O*NET-SOC Code', 'Element ID', 'Element Name', 'Scale ID', 'Data Value', 'N', 'Standard Error', 'Lower CI Bound', 'Upper CI Bound', 'Recommend Suppress', 'Not Relevant', 'Date', 'Domain Source']
